# Data Mining - What should the next Champion look like?

## Import

In [ ]:
# 📌 Installiere fehlende Bibliotheken, falls notwendig
import sys
!{sys.executable} -m pip install xgboost seaborn scikit-learn pandas numpy matplotlib

# 📌 Import notwendiger Bibliotheken
import pandas as pd  # Datenverarbeitung
import numpy as np  # Numerische Berechnungen
import matplotlib.pyplot as plt  # Visualisierung
import seaborn as sns  # Erweiterte Visualisierung
from sklearn.model_selection import train_test_split  # Aufteilung in Trainings- & Testdaten
from sklearn.ensemble import RandomForestClassifier  # Machine Learning Modell
from sklearn.linear_model import LogisticRegression  # Lineares Modell für Vergleich
from sklearn.svm import SVC  # Support Vector Machine Modell
from xgboost import XGBClassifier  # Boosted Trees Modell
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report  # Modellbewertung
from sklearn.preprocessing import StandardScaler, OneHotEncoder  # Skalierung & Encoding
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV  # Hyperparameter-Tuning





📊 Erste 5 Zeilen des Datensatzes:
     Name     Class Role Tier  Score  Trend   Win %  Role % Pick %   Ban %  \
0  Aatrox   Fighter  TOP    A  58.25   6.52  49.97%  94.62%  4.43%   2.03%   
1    Ahri      Mage  MID    A  53.21  -0.24  49.93%  93.47%  4.62%   1.04%   
2   Akali  Assassin  MID    S  65.30   6.51  48.59%  65.65%  8.16%  12.88%   
3   Akali  Assassin  TOP    A  57.87   3.34  48.57%  34.06%  4.24%  12.88%   
4  Akshan  Marksman  MID    S  59.85   0.65  51.46%  58.01%  4.83%  21.91%   

    KDA  
0  1.97  
1  2.56  
2  2.34  
3  2.04  
4  2.23  

📌 Datensatz-Übersicht:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232 entries, 0 to 231
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Name    232 non-null    object 
 1   Class   231 non-null    object 
 2   Role    232 non-null    object 
 3   Tier    232 non-null    object 
 4   Score   232 non-null    float64
 5   Trend   232 non-null    float64
 6   Win %  


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# 📌 Datensatz laden
df = pd.read_csv('League of Legends Champion Stats 12.1.csv', sep=';')


In [5]:
# 📌 Erste Ansicht der Daten
print("📊 Erste 5 Zeilen des Datensatzes:")
print(df.head())




📊 Erste 5 Zeilen des Datensatzes:
     Name     Class Role Tier  Score  Trend   Win %  Role % Pick %   Ban %  \
0  Aatrox   Fighter  TOP    A  58.25   6.52  49.97%  94.62%  4.43%   2.03%   
1    Ahri      Mage  MID    A  53.21  -0.24  49.93%  93.47%  4.62%   1.04%   
2   Akali  Assassin  MID    S  65.30   6.51  48.59%  65.65%  8.16%  12.88%   
3   Akali  Assassin  TOP    A  57.87   3.34  48.57%  34.06%  4.24%  12.88%   
4  Akshan  Marksman  MID    S  59.85   0.65  51.46%  58.01%  4.83%  21.91%   

    KDA  
0  1.97  
1  2.56  
2  2.34  
3  2.04  
4  2.23  


In [6]:
print("\n📌 Datensatz-Übersicht:")
df.info()



📌 Datensatz-Übersicht:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 232 entries, 0 to 231
Data columns (total 11 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Name    232 non-null    object 
 1   Class   231 non-null    object 
 2   Role    232 non-null    object 
 3   Tier    232 non-null    object 
 4   Score   232 non-null    float64
 5   Trend   232 non-null    float64
 6   Win %   232 non-null    object 
 7   Role %  232 non-null    object 
 8   Pick %  232 non-null    object 
 9   Ban %   232 non-null    object 
 10  KDA     232 non-null    float64
dtypes: float64(3), object(8)
memory usage: 20.1+ KB


In [7]:
print("\n📌 Statistische Beschreibung der numerischen Spalten:")
print(df.describe())


📌 Statistische Beschreibung der numerischen Spalten:
            Score       Trend         KDA
count  232.000000  232.000000  232.000000
mean    50.000216   -0.264483    2.331336
std     16.532143    6.439499    0.415322
min     11.030000  -20.170000    1.450000
25%     38.852500   -3.460000    2.040000
50%     46.560000   -0.815000    2.300000
75%     59.247500    2.100000    2.542500
max     94.230000   44.710000    4.110000


## Pre-processing

In [ ]:
# 📌 Entfernen von Sonderzeichen in Prozentwerten & Konvertierung in Float
df['Win %'] = df['Win %'].str.replace('%', '').astype(float)
df['Role %'] = df['Role %'].str.replace('%', '').astype(float)
df['Pick %'] = df['Pick %'].str.replace('%', '').astype(float)
df['Ban %'] = df['Ban %'].str.replace('%', '').astype(float)




Fehlende Werte pro Spalte:
 Name      0
Class     1
Role      0
Tier      0
Score     0
Trend     0
Win %     0
Role %    0
Pick %    0
Ban %     0
KDA       0
dtype: int64
✅ Feature Engineering abgeschlossen. Daten bereit für das Modell.
Feature-Namen: ['Win %', 'Pick %', 'Ban %', 'Score', 'KDA', 'Class_Assassin', 'Class_Fighter', 'Class_Mage', 'Class_Marksman', 'Class_Support', 'Class_Tank', 'Role_ADC', 'Role_JUNGLE', 'Role_MID', 'Role_SUPPORT', 'Role_TOP']


In [9]:
# 📌 Überprüfung auf fehlende Werte
missing_values = df.isnull().sum()
print("Fehlende Werte pro Spalte:\n", missing_values)



Fehlende Werte pro Spalte:
 Name      0
Class     0
Role      0
Tier      0
Score     0
Trend     0
Win %     0
Role %    0
Pick %    0
Ban %     0
KDA       0
dtype: int64


In [10]:
# Falls fehlende Werte existieren, entfernen wir sie
df.dropna(inplace=True)



In [11]:
# 📌 Trennen von numerischen & kategorialen Variablen für Feature Engineering
numerical_features = ['Win %', 'Pick %', 'Ban %', 'Score', 'KDA']
categorical_features = ['Class', 'Role']



In [12]:
# 📌 Skalierung der numerischen Features (StandardScaler für Normalverteilung)
numerical_transformer = StandardScaler()



In [13]:
# 📌 One-Hot-Encoding für kategoriale Variablen
categorical_transformer = OneHotEncoder(handle_unknown='ignore')



In [14]:
# 📌 Feature Engineering Pipeline für Preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])



In [15]:
# 📌 Transformation der Daten
df_processed = preprocessor.fit_transform(df)



In [16]:
# 📌 Feature-Namen extrahieren
feature_names = numerical_features + list(preprocessor.named_transformers_['cat'].get_feature_names_out())

print("✅ Feature Engineering abgeschlossen. Daten bereit für das Modell.")
print(f"Feature-Namen: {feature_names}")

✅ Feature Engineering abgeschlossen. Daten bereit für das Modell.
Feature-Namen: ['Win %', 'Pick %', 'Ban %', 'Score', 'KDA', 'Class_Assassin', 'Class_Fighter', 'Class_Mage', 'Class_Marksman', 'Class_Support', 'Class_Tank', 'Role_ADC', 'Role_JUNGLE', 'Role_MID', 'Role_SUPPORT', 'Role_TOP']


## Modellauswahl und -beschreibung

In [17]:
# 📌 Zielvariable erstellen (Underrepresentation)
df['Underrepresented'] = (df['Role %'] < df['Role %'].median()).astype(int)

# 📌 Train-Test-Split (80% Training, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(df_processed, df['Underrepresented'], test_size=0.2, random_state=42)

# 📌 Definiere mehrere Modelle für den Vergleich
models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'XGBoost': XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
    'Logistic Regression': LogisticRegression(),
    'SVM': SVC()
}

# 📌 Training & Evaluation
results = {}
for name, model in models.items():
    model.fit(X_train, y_train)  # Trainiere das Modell
    y_pred = model.predict(X_test)  # Teste das Modell
    acc = accuracy_score(y_test, y_pred)  # Berechne Genauigkeit
    results[name] = acc  # Speichere Ergebnisse
    print(f"🔥 {name} Accuracy: {acc:.4f}")

# 📌 Vergleich der Modellgenauigkeiten
print("\n📊 Modellvergleich:")
for model, score in results.items():
    print(f"{model}: {score:.4f}")

# 📌 Bestes Modell auswählen
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]
print(f"\n✅ Bestes Modell: {best_model_name} mit Accuracy von {results[best_model_name]:.4f}")


🔥 Random Forest Accuracy: 0.7447
🔥 XGBoost Accuracy: 0.8085
🔥 Logistic Regression Accuracy: 0.7447
🔥 SVM Accuracy: 0.7234

📊 Modellvergleich:
Random Forest: 0.7447
XGBoost: 0.8085
Logistic Regression: 0.7447
SVM: 0.7234

✅ Bestes Modell: XGBoost mit Accuracy von 0.8085


c:\Users\maxtr\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:158: UserWarning: [19:07:33] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


### Hyperparameter Tuning

In [18]:
# 📌 Parameter für GridSearchCV definieren
param_grid = {
    'n_estimators': [50, 100, 200],  # Anzahl der Bäume
    'learning_rate': [0.01, 0.1, 0.2],  # Lernrate
    'max_depth': [3, 5, 7]  # Maximale Baumtiefe
}

# 📌 GridSearch mit 5-facher Cross-Validation
grid_search = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='logloss'),
                           param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train, y_train)

# 📌 Bestes Modell & Parameter ausgeben
best_xgb = grid_search.best_estimator_
print(f"\n✅ Beste Parameter für XGBoost: {grid_search.best_params_}")
print(f"🔥 Beste Accuracy nach Tuning: {grid_search.best_score_:.4f}")



✅ Beste Parameter für XGBoost: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 50}
🔥 Beste Accuracy nach Tuning: 0.7556


c:\Users\maxtr\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:158: UserWarning: [19:08:54] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


In [19]:
# 📌 Bestes XGBoost-Modell aus GridSearch nutzen
y_pred_best = best_xgb.predict(X_test)

# 📌 Finale Accuracy auf dem Testset berechnen
final_accuracy = accuracy_score(y_test, y_pred_best)

print(f"🔥 Finale Accuracy des getunten XGBoost-Modells: {final_accuracy:.4f}")


🔥 Finale Accuracy des getunten XGBoost-Modells: 0.8085
